In [5]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from scikeras.wrappers import KerasClassifier # Changed import
import time
import pandas as pd
from tensorflow.keras.optimizers import Adam, RMSprop #Added optimizer

# Function to extract MFCCs from audio files
def extract_mfccs(file_path):
    try:
        y, sr = librosa.load(file_path, duration=1)  # Load 1-second audio
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=60)
        return np.mean(mfccs.T, axis=0)  # Average MFCCs over time
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Data Loading and Feature Extraction
male_folder = "/home/feliciano/adult_lobsters"  # Replace with your actual male folder name if different
female_folder = "/home/feliciano/juvenile_lobsters" # Replace with your actual female folder name if different

data = []
labels = []

for filename in os.listdir(male_folder):
    file_path = os.path.join(male_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("adult")

for filename in os.listdir(female_folder):
    file_path = os.path.join(female_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("juvenile")

if not data:
    print("No audio files found in the specified folders.")
    exit()

X = np.array(data)
y = np.array(labels)

# Label Encoding and Data Scaling
le = LabelEncoder()
y = le.fit_transform(y)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# PCA
n_components = min(X_train.shape[1], 30)  # You can adjust this
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Original feature shape: {X_train.shape}")
print(f"Shape after PCA: {X_train_pca.shape}")
print(f"Explained variance ratio of the first {n_components} components: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {sum(pca.explained_variance_ratio_):.4f}")

# Reshape data for CNN after PCA
X_train_pca_reshaped = X_train_pca.reshape(X_train_pca.shape[0], X_train_pca.shape[1], 1)
X_test_pca_reshaped = X_test_pca.reshape(X_test_pca.shape[0], X_test_pca.shape[1], 1)

# Function to create the 1D-CNN model with 2 layers (for KerasClassifier)
def create_model(optimizer='adam', filters_1=64, kernel_size_1=3, pool_size_1=2, filters_2=128, kernel_size_2=3, pool_size_2=2, dense_units=128):
    model = Sequential()
    model.add(Conv1D(filters_1, kernel_size_1, activation='relu', input_shape=(X_train_pca_reshaped.shape[1], 1)))
    model.add(MaxPooling1D(pool_size_1))
    model.add(Conv1D(filters_2, kernel_size_2, activation='relu'))
    model.add(MaxPooling1D(pool_size_2))
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Binary classification
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Wrap the Keras model for use with scikit-learn's GridSearchCV
cnn_model = KerasClassifier(model=create_model, verbose=0)

# Define the parameter grid for GridSearchCV
param_grid = {
    'optimizer': [Adam, RMSprop],
    'model__filters_1': [32, 64, 128],
    'model__kernel_size_1': [3, 5],
    'model__pool_size_1': [2, 3],
    'model__filters_2': [64, 128, 256],
    'model__kernel_size_2': [3, 5],
    'model__pool_size_2': [2, 3],
    'model__dense_units': [64, 128, 256],
    'epochs': [10, 20],
    'batch_size': [32, 64],
}

# Perform Grid Search
grid_search = GridSearchCV(estimator=cnn_model,
                           param_grid=param_grid,
                           cv=3,  # Reduced CV for speed
                           scoring='accuracy',
                           verbose=2,
                           n_jobs=-1)

start_time_grid = time.time()
grid_search.fit(X_train_pca_reshaped, y_train)
end_time_grid = time.time()
grid_search_time = (end_time_grid - start_time_grid)

best_cnn_model = grid_search.best_estimator_

y_pred_grid = best_cnn_model.predict(X_test_pca_reshaped)
y_pred_proba_grid = best_cnn_model.predict_proba(X_test_pca_reshaped) # Changed this line
y_pred_proba_grid = y_pred_proba_grid[:, 1]
# Calculate Metrics
accuracy_grid = accuracy_score(y_test, y_pred_grid)
precision_grid = precision_score(y_test, y_pred_grid)
recall_grid = recall_score(y_test, y_pred_grid)
f1_grid = f1_score(y_test, y_pred_grid)
auc_roc_grid = roc_auc_score(y_test, y_pred_proba_grid)

# Print Cross-Validation Results
print("\nCross-Validation Results:")
cv_results_df = pd.DataFrame(grid_search.cv_results_)
print(cv_results_df[[
    'param_optimizer', 'param_model__filters_1', 'param_model__kernel_size_1', 'param_model__pool_size_1',
    'param_model__filters_2', 'param_model__kernel_size_2', 'param_model__pool_size_2',
    'param_model__dense_units', 'param_epochs', 'param_batch_size', 'mean_test_score',
    'std_test_score', 'rank_test_score'
]])

# Print Best Model Results
print("\nBest 1D-CNN (2 Layers) Model Performance after PCA and Grid Search:")
print(f"  Best Parameters: {grid_search.best_params_}")
print(f"  Accuracy: {accuracy_grid:.4f}")
print(f"  Precision: {precision_grid:.4f}")
print(f"  Recall: {recall_grid:.4f}")
print(f"  F1-Score: {f1_grid:.4f}")
print(f"  AUC-ROC: {auc_roc_grid:.4f}")
print(f"  Grid Search Time (s): {grid_search_time:.4f}")


Original feature shape: (5845, 60)
Shape after PCA: (5845, 30)
Explained variance ratio of the first 30 components: [0.19666824 0.10417445 0.08138078 0.05794707 0.05558503 0.04463546
 0.03685725 0.03339994 0.027507   0.02429197 0.02156267 0.01988868
 0.01812808 0.01659644 0.01491982 0.0143873  0.01328331 0.01289562
 0.01213517 0.01171574 0.01148071 0.01058967 0.00954135 0.0094476
 0.00910745 0.00885667 0.00828926 0.00798621 0.00753847 0.00688206]
Total explained variance: 0.9077
Fitting 3 folds for each of 3456 candidates, totalling 10368 fits


2025-05-12 12:29:34.568246: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747049374.595439 2215052 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747049374.603765 2215052 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-12 12:29:34.628686: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-12 12:29:34.907733: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT f

[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=3, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  10.4s
[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  10.9s
[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=5, model__pool_size_1=3, model__pool_size_2=3, optimizer=<class 'keras.src.optimizers.rmsprop.RMSprop'>; total time=  10.3s
[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=5, model__kernel_size_2=3, model__pool_size_1=3, model__pool_size_2=2, optimizer=<cl

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model i

[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=3, model__pool_size_1=2, model__pool_size_2=3, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  10.4s
[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  11.0s
[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=5, model__kernel_size_2=3, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  11.4s
[CV] END batch_size=32, epochs=10, model__dense_units=64, model__filters_1=32, model__filters_2=64, model__kernel_size_1=5, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'k

...

[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=64, model__filters_2=256, model__kernel_size_1=3, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  36.0s
[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=64, model__filters_2=256, model__kernel_size_1=5, model__kernel_size_2=3, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.rmsprop.RMSprop'>; total time=  39.0s
[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=64, model__filters_2=256, model__kernel_size_1=5, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.adam.Adam'>; total time=  29.9s
[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=64, model__filters_2=256, model__kernel_size_1=5, model__kernel_size_2=5, model__pool_size_1=3, model__pool_size_2=3, optim

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model i

[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=64, model__filters_2=256, model__kernel_size_1=5, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.rmsprop.RMSprop'>; total time=  37.7s
[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=128, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=3, model__pool_size_1=2, model__pool_size_2=3, optimizer=<class 'keras.src.optimizers.rmsprop.RMSprop'>; total time=  24.1s
[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=128, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=5, model__pool_size_1=2, model__pool_size_2=2, optimizer=<class 'keras.src.optimizers.rmsprop.RMSprop'>; total time=  24.4s
[CV] END batch_size=64, epochs=20, model__dense_units=256, model__filters_1=128, model__filters_2=64, model__kernel_size_1=3, model__kernel_size_2=5, model__pool_size_1=3, model__pool_siz

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Cross-Validation Results:
                                     param_optimizer param_model__filters_1  \
0           <class 'keras.src.optimizers.adam.Adam'>                     32   
1     <class 'keras.src.optimizers.rmsprop.RMSprop'>                     32   
2           <class 'keras.src.optimizers.adam.Adam'>                     32   
3     <class 'keras.src.optimizers.rmsprop.RMSprop'>                     32   
4           <class 'keras.src.optimizers.adam.Adam'>                     32   
...                                              ...                    ...   
3451  <class 'keras.src.optimizers.rmsprop.RMSprop'>                    128   
3452        <class 'keras.src.optimizers.adam.Adam'>                    128   
3453  <class 'keras.src.optimizers.rmsprop.RMSprop'>                    128   
3454        <class 'keras.src.optimizers.adam.Adam'>                    128   
3455  <class 'keras.src.optimizers.rmsprop.RMSprop'>                    128   

     param_model__kernel